# Notebook 1: News/Text Modality - FinBERT Model Comparison
### Simple comparison between ProsusAI/finbert vs yiyanghkust/finbert-tone

1. ProsusAI/finbert: trained on news and analyst reports; labels either positive, negative or neutral
2. yiyanghkust/finbert-tone: trained on forward-looking financial statement like earning calls; same labels as above

### Step 0 - Basic check that we are in virtual environment

In [2]:
import sys
print(sys.executable)

C:\Users\KJ\Documents\School\FYP\Feature Prototype ENV\.venv\Scripts\python.exe


### Step 1 - Install dependencies and Imports

In [3]:
# Install dependencies
# * my dependencies are already installed directly in venv through terminal so it is commented out

# %pip install transformers torch --quiet

In [7]:
# Imports
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BertTokenizer, BertForSequenceClassification, pipeline
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

### Step 2 - Mock headlines containing bullish, bearish and neutral sounding news headline
- only headlines are used as both BERT models are only commonly benchmarked on short financial text
- not using body also means there are lesser noise and lower computational cost from my side

In [8]:
# Sample headlines
headlines = [
    "Gold holds losses as Iran impasse keeps rate hike bets high",
    "Trump says he'll let Warsh 'do what he wants to do' with interest rates. It's a remark that Fed watchers have been bracing for",
    "China’s Xi says the basis of trust between Beijing and Moscow is becoming increasingly solid",
    "EU reaches provisional trade deal with US, seeks to head off fresh Trump tariffs",
    "US 30-Year Yield Hits Highest Since 2007 on Inflation Angst",
    "Bitcoin surges past $70,000 on renewed institutional buying interest",
    "Asian markets mixed ahead of key US inflation data release"
]

### Step 3 - Load both models
- pipeline will automatically check if model exist else downloads it from hugging face
- usage is manual load from the documentation found from https://huggingface.co/yiyanghkust/finbert-tone

In [9]:
print("Loading ProsusAI/finbert...")
prosus_model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
prosus_tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
prosus_pipeline = pipeline("sentiment-analysis", model=prosus_model, tokenizer=prosus_tokenizer)


# We will be manual loading finbert-tone model as Automodel does not work for this
print("Loading yiyanghkust/finbert-tone...")
finbert_tone_model = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")
finbert_tone_tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-tone")
finbert_tone_pipeline = pipeline("sentiment-analysis", model=finbert_tone_model, tokenizer=finbert_tone_tokenizer)

print("Both model has been loaded")

Loading ProsusAI/finbert...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading yiyanghkust/finbert-tone...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both model has been loaded


### Step 4 - Run both models

In [13]:
# Run inference and collect the result
def run_model(pipe, headlines):
    results = []
    for text in headlines:
        output = pipe(text)
        # Sort so highest score is first
        output_sorted = sorted(output, key=lambda x: x['score'], reverse=True)
        top = output_sorted[0]
        results.append({
            "headline": text,
            "label": top['label'],
            "confidence": round(top['score'], 4)
        })
    return results

# Run both
results_prosus = run_model(prosus_pipeline, headlines)
results_tone = run_model(finbert_tone_pipeline, headlines)

### Step 5 - Compare results of both models

In [14]:
# Compare both in a table
rows = []
for r1, r2 in zip(results_prosus, results_tone):
    rows.append({
        "Headline": r1['headline'][:60] + "...",
        "ProsusAI Label": r1['label'],
        "ProsusAI Conf": r1['confidence'],
        "FinBERT-Tone Label": r2['label'],
        "FinBERT-Tone Conf": r2['confidence'],
        "Agreement": "T" if r1['label'].lower() == r2['label'].lower() else "F"
    })

# Show result
df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', None)
df

,Headline,ProsusAI Label,ProsusAI Conf,FinBERT-Tone Label,FinBERT-Tone Conf,Agreement
0,Gold holds losses as Iran impasse keeps rate hike bets high...,negative,0.6100,Negative,0.9998,T
1,Trump says he'll let Warsh 'do what he wants to do' with int...,neutral,0.6295,Neutral,0.9595,T
2,China’s Xi says the basis of trust between Beijing and Mosco...,neutral,0.5389,Positive,1.0000,F
3,"EU reaches provisional trade deal with US, seeks to head off...",positive,0.9201,Neutral,0.9994,F
4,US 30-Year Yield Hits Highest Since 2007 on Inflation Angst...,positive,0.8012,Positive,1.0000,T
5,"Bitcoin surges past $70,000 on renewed institutional buying ...",positive,0.9397,Positive,1.0000,T
6,Asian markets mixed ahead of key US inflation data release...,negative,0.8807,Negative,0.7287,T


### Step 6 - Convert to direction signal for downstream fusion
- labels used are: positve = 1, negative = -1, neutral = 0

In [15]:
# Maps sentiment label to numeric signal so that the final fusion stage can consume later on
LABEL_MAP = {
    "positive": 1,
    "negative": -1,
    "neutral": 0
}

def to_signal(results, model_name):
    print(f"\n{model_name} Directional Signals:")
    for r in results:
        signal = LABEL_MAP.get(r['label'].lower(), 0)
        print(f"  [{signal:+d}] ({r['label']}, {r['confidence']}) | {r['headline'][:60]}"+ "...")

to_signal(results_prosus, "ProsusAI/finbert")
to_signal(results_tone, "yiyanghkust/finbert-tone")


ProsusAI/finbert Directional Signals:
  [-1] (negative, 0.61) | Gold holds losses as Iran impasse keeps rate hike bets high...
  [+0] (neutral, 0.6295) | Trump says he'll let Warsh 'do what he wants to do' with int...
  [+0] (neutral, 0.5389) | China’s Xi says the basis of trust between Beijing and Mosco...
  [+1] (positive, 0.9201) | EU reaches provisional trade deal with US, seeks to head off...
  [+1] (positive, 0.8012) | US 30-Year Yield Hits Highest Since 2007 on Inflation Angst...
  [+1] (positive, 0.9397) | Bitcoin surges past $70,000 on renewed institutional buying ...
  [-1] (negative, 0.8807) | Asian markets mixed ahead of key US inflation data release...

yiyanghkust/finbert-tone Directional Signals:
  [-1] (Negative, 0.9998) | Gold holds losses as Iran impasse keeps rate hike bets high...
  [+0] (Neutral, 0.9595) | Trump says he'll let Warsh 'do what he wants to do' with int...
  [+1] (Positive, 1.0) | China’s Xi says the basis of trust between Beijing and Mosco...
  [+0] 

### Step 7 - Final observation + decision

Observation:
- Both models produced similar sentiment classifications for most headlines, with a 5/7 agreement rate
- FinBERT-Tone generally returned higher confidence scores, with several predictions at or near 1.0 confidence
- ProsusAI produced a wider spread of confidence values across different headlines

Chosen model for pipeline and reasoning: **ProsusAI/finbert**
- Both models are finance-domain BERT variants and financial sentiment classifiers, but were fine-tuned on slightly different datasets/objectives (FinBERT -> broader financial communications such as reports, filings, and earnings-related text; FinBERT-Tone -> financial tone analysis and PhraseBank-style sentiment annotation)
- Both models demonstrated acceptable performance across the sampled financial headlines used in this prototype
- ProsusAI/finbert was selected due to its more balanced confidence distribution across different headline types